In this notebook, I verify the desired properties of tje extended transformer model I generate

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import random

import torch

from extend_model import DecodeExtendedTokenizer, extend_model

In [2]:
MODEL_NAME = "Qwen/Qwen3.5-9B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
extended_tokenizer = DecodeExtendedTokenizer(
    base_tokenizer = tokenizer,
    new_token_strs = ["blackmail"]
)

In [3]:
vocab_size = len(tokenizer)

# Test 1: no change to encoding

input = "blackmail"
print(tokenizer.encode(input))
print(extended_tokenizer.encode(input))

# Test 2: no change to decoding for general token

random_id = random.randint(0, len(tokenizer) - 1)

print(tokenizer.decode([random_id]))
print(extended_tokenizer.decode([random_id]))

# Test 3: able to decode new tokens

new_id = len(tokenizer)

print(extended_tokenizer.decode([new_id]))

[11124, 3585]
[11124, 3585]
 spawned
 spawned
blackmail


In [4]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

In [5]:
model.eval()

prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

with torch.no_grad():
    out = model(input_ids)

print(model.get_output_embeddings().weight.dtype)

logits = out.logits  # [batch, seq_len, vocab_size]
print(logits.shape)

torch.bfloat16
torch.Size([1, 5, 248320])


In [6]:
new_weights = torch.zeros((2, 4096), dtype=torch.bfloat16)

extend_model(model, new_weights)


In [7]:
model.eval()

print(model.get_output_embeddings().weight.dtype)

prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

with torch.no_grad():
    out = model(input_ids)

logits = out.logits  # [batch, seq_len, vocab_size]
print(logits.shape)

torch.bfloat16
torch.Size([1, 5, 248322])
